In [6]:
import sys
sys.path.append("../src/")
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
from relaxers.quadratic_model_lin_approx import quadratic_model_lin_approx # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_no_surrogate import quadratic_lin_approx_no_surrogate # type: ignore[reportMissingImports]
from relaxers.quadratic_lin_approx_nosur_refine import quadratic_lin_approx_no_surrogate_refine # type: ignore[reportMissingImports]
from testers.classical_solve_lin_program import solve_lp_return_x # type: ignore[reportMissingImports]
from testers.classical_solve_quad_program import solve_qcp # type: ignore[reportMissingImports]
from testers.condition_number_lin_program import condition_number_nes_basic # type: ignore[reportMissingImports]

import matplotlib.pyplot as plt
import numpy as np

In [8]:
LIN_APPROX = 5

# Whether to do an inner or outer approximation
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [9]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                            f_name="model11_quad_reform_nofix.json", endpoints=False)
val, time, x = solve_lp_return_x()
print(x)
quad, q_time = solve_qcp(f_name="model11_quad_reform_nofix.json")

print(condition_number_nes_basic())
print(f"Error: {val - quad}")

[160.0, 163.8888888888889, 140.0, 100.0, 63.888888888888914, 136.1111111111111, 120.0, 80.0, 160.0, 200.0, 2472.5063217220954, 1846.039152232345, 2086.9880635745567, 3388.537657392857, 2192.7343219018408, 2891.7574274575436, 1600.0, 1773.7544788296816, 3142.769767823801, 3489.736200156586]
2083.2261708373176
Error: -914.1612911895768


In [10]:
# In this case there are 10 quadratic variables
# Define a set of functions for each

# TODO: It should intelligently search for the quadratic variables, not just assume they're at the start

iters = 5

for i in range(iters):
    print(f"Iteration {i}")
    quads = x[:10]
    points_functions = []


    # def np_uniform_add_q_factory(q):
    #     def np_uniform_add_q(lower, upper, num):
    #             uniform = np.linspace(lower, upper, num)
    #             idx = np.searchsorted(uniform, q)
    #             return np.insert(uniform, idx, q)
    #     return np_uniform_add_q
    
    def np_uniform_add_q_factory(q):
        def np_uniform_add_q(lower, upper, num):
                return [q]
        return np_uniform_add_q

    for q in quads:
        points_functions.append(np_uniform_add_q_factory(q))

    # quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + 1, OUTER_APPROXIMATION, points_function=points_functions,
    #                                             remove_division=REMOVE_DIVISION, f_name="model11_quad_reform_nofix.json")

    quadratic_lin_approx_no_surrogate_refine(1, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION,
                                                f_name="model11_quad_reform_nofix.json")
    val, time, x = solve_lp_return_x()
    # print(f"x: {x}")
    # print(f"val: {val}")

print(x)
print(f"Error: {val - quad}")

Iteration 0
Iteration 1
Iteration 2
Iteration 3
Iteration 4
[160.00000000000003, 146.41016151377545, 140.0, 100.0, 46.41016151377543, 153.58983848622458, 120.0, 80.0, 160.00000000000003, 200.0, 2868.535985170749, 2361.43030147469, 2310.475842065843, 3751.4035341088515, 2461.891524262877, 3332.209308379107, 1856.2773878274704, 2124.8622794296098, 3410.7455823639352, 3600.0]
Error: -1.5775090105307754e-06
